# 12 — Soft Actor-Critic (SAC)

## Learning Objectives
1. Understand the entropy-regularised objective J(pi) and why it improves exploration
2. Implement twin critics and explain how they reduce Q-value overestimation
3. Build SAC with automatic temperature tuning on a continuous control task
4. Compare SAC exploration behaviour against PPO across temperature values


In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import time
from collections import deque
from typing import Tuple, List, Dict

np.random.seed(42)

print("NumPy:", np.__version__)
print("No gym/torch required — all environments hand-coded in numpy")


## Level 1: Entropy Regularisation — Core Mechanics

SAC maximises the entropy-augmented return:
J(pi) = E[sum_t gamma^t (r_t + alpha * H(pi(.|s_t)))]

The temperature alpha controls exploration:
- alpha=0   -> pure reward maximisation (can be brittle)
- alpha=1   -> equal weight to entropy (maximum exploration)
- alpha~0.2 -> typical production default

Here we isolate the entropy computation and show its effect on policy diversity.


In [ ]:
# --- Level 1: Entropy bonus and its effect on policy ---

def entropy(probs: np.ndarray) -> float:
    """Shannon entropy H(pi) = -sum p_i * log(p_i)."""
    probs = np.clip(probs, 1e-10, 1.0)
    return float(-np.sum(probs * np.log(probs)))


def softmax(logits: np.ndarray, temp: float = 1.0) -> np.ndarray:
    """Softmax with temperature scaling."""
    lg = logits / temp
    lg = lg - lg.max()
    p = np.exp(lg)
    return p / p.sum()


# Show how temperature controls policy entropy
logits_example = np.array([2.0, 1.0, 0.5, 0.1])
temps = [0.1, 0.5, 1.0, 2.0, 5.0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
entropy_vals = []
for temp in temps:
    p = softmax(logits_example, temp)
    h = entropy(p)
    entropy_vals.append(h)

axes[0].plot(temps, entropy_vals, "o-", color="steelblue", linewidth=2, markersize=8)
axes[0].set_xlabel("Temperature (alpha)")
axes[0].set_ylabel("Policy Entropy H(pi)")
axes[0].set_title("Higher Temperature -> Higher Entropy (More Exploration)")
axes[0].grid(True, alpha=0.3)

# Show policy distributions at different temperatures
for i, temp in enumerate([0.1, 1.0, 5.0]):
    p = softmax(logits_example, temp)
    axes[1].bar(np.arange(4) + i*0.25, p, width=0.25, label=f"T={temp}", alpha=0.8)
axes[1].set_xlabel("Action"); axes[1].set_ylabel("Probability")
axes[1].set_title("Action Distribution at Different Temperatures")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/tmp/sac_entropy.png", dpi=80, bbox_inches="tight")
plt.show()

print("Temperature -> Entropy:")
for t, h in zip(temps, entropy_vals):
    print(f"  T={t:.1f}: H={h:.3f}")
print()
print("Insight: SAC uses alpha as a learnable temperature to balance exploration/exploitation")


## Level 2: Full SAC with Twin Critics and Auto-Temperature

We implement SAC on a 1D particle task (x_next = x + a + noise, goal: reach x=0).

Key SAC components:
- **Twin critics Q1, Q2**: use min(Q1, Q2) for target to prevent overestimation
- **Soft Bellman target**: target_Q = r + gamma * (min(Q1', Q2') - alpha * log_pi(a'|s'))
- **Auto-tuning alpha**: gradient of E[-alpha * log_pi - alpha * H_target] wrt log_alpha
- **Replay buffer**: off-policy learning — SAC reuses past experience


In [ ]:
# === 1D Particle Environment ===

def particle_step(state, action, dt=0.1):
    """x_next = x + a + noise. Goal: reach x=0."""
    action = np.clip(action, -1.0, 1.0)
    noise = np.random.normal(0, 0.01)
    state_new = np.clip(state + action * dt + noise, -3.0, 3.0)
    reward = -abs(state_new)   # dense negative-distance reward
    done = bool(abs(state_new) < 0.05)
    return state_new, reward, done


def particle_reset():
    return float(np.random.uniform(-2.0, 2.0))


# === Replay Buffer ===

class ReplayBuffer:
    def __init__(self, capacity=20000):
        self.cap = capacity; self.ptr = 0; self.size = 0
        self.S  = np.zeros((capacity, 1)); self.A = np.zeros((capacity, 1))
        self.R  = np.zeros(capacity); self.NS = np.zeros((capacity, 1))
        self.D  = np.zeros(capacity)

    def store(self, s, a, r, ns, d):
        i = self.ptr % self.cap
        self.S[i] = s; self.A[i] = a; self.R[i] = r; self.NS[i] = ns; self.D[i] = float(d)
        self.ptr += 1; self.size = min(self.ptr, self.cap)

    def sample(self, bs=256):
        idx = np.random.choice(self.size, bs, replace=False)
        return dict(s=self.S[idx], a=self.A[idx], r=self.R[idx], ns=self.NS[idx], d=self.D[idx])


# === Gaussian Actor ===

class GaussianActor:
    """pi(a|s) = N(W*s+b, exp(log_std)^2)."""
    def __init__(self, lr=3e-3):
        self.W = np.zeros((1, 1)); self.b = np.zeros(1)
        self.log_std = np.array([-0.5]); self.lr = lr

    def mean(self, s): return float(self.W[0, 0] * s + self.b[0])
    def std(self): return float(np.exp(np.clip(self.log_std[0], -4, 1)))

    def sample(self, s):
        mu, sigma = self.mean(s), self.std()
        noise = np.random.normal()
        a = mu + sigma * noise
        lp = -0.5 * noise**2 - np.log(sigma) - 0.5 * np.log(2 * np.pi)
        return np.clip(a, -1.0, 1.0), float(lp)

    def log_prob(self, s, a):
        mu, sigma = self.mean(s), self.std()
        return float(-0.5 * ((a - mu) / sigma)**2 - np.log(sigma) - 0.5 * np.log(2 * np.pi))


# === Linear Q-function ===

class QFunction:
    """Q(s, a) = W_sa . [s, a] + b."""
    def __init__(self, lr=1e-2):
        self.W = np.zeros(2); self.b = 0.0; self.lr = lr

    def predict_batch(self, S, A):
        SA = np.concatenate([S, A], axis=1)
        return SA @ self.W + self.b

    def predict(self, s, a):
        return float(np.array([s, a]) @ self.W + self.b)

    def update(self, S, A, targets):
        SA = np.concatenate([S, A], axis=1)
        err = SA @ self.W + self.b - targets
        self.W -= self.lr * SA.T @ err / len(err)
        self.b -= self.lr * err.mean()
        return float((err**2).mean())


# === SAC Agent ===

class SACAgent:
    """SAC with twin critics and auto-tuned temperature."""
    def __init__(self, auto_alpha=True, alpha=0.2):
        self.gamma = 0.99; self.tau = 0.005
        self.alpha = alpha; self.log_alpha = np.log(alpha)
        self.auto_alpha = auto_alpha
        self.target_entropy = -1.0; self.alpha_lr = 5e-4

        self.actor = GaussianActor(lr=3e-3)
        self.q1 = QFunction(lr=1e-2); self.q2 = QFunction(lr=1e-2)
        self.q1t = QFunction(); self.q2t = QFunction()
        self.q1t.W = self.q1.W.copy(); self.q2t.W = self.q2.W.copy()

    def act(self, s):
        a, _ = self.actor.sample(float(s))
        return a

    def update(self, batch):
        S, A, R, NS, D = batch["s"], batch["a"], batch["r"], batch["ns"], batch["d"]

        # Compute soft targets
        na_list, nlp_list = [], []
        for i in range(len(NS)):
            na, nlp = self.actor.sample(float(NS[i, 0]))
            na_list.append(na); nlp_list.append(nlp)
        NA = np.array(na_list).reshape(-1, 1)
        nlps = np.array(nlp_list)

        # Use min of twin critics to prevent overestimation
        q1_next = self.q1t.predict_batch(NS, NA)
        q2_next = self.q2t.predict_batch(NS, NA)
        targets = R + self.gamma * (1 - D) * (np.minimum(q1_next, q2_next) - self.alpha * nlps)

        # Update critics
        self.q1.update(S, A, targets); self.q2.update(S, A, targets)

        # Update actor
        for i in range(len(S)):
            a_new, lp = self.actor.sample(float(S[i, 0]))
            q_val = min(self.q1.predict(S[i, 0], a_new), self.q2.predict(S[i, 0], a_new))
            # Gradient: maximize Q - alpha * log_pi
            mu, sigma = self.actor.mean(S[i, 0]), self.actor.std()
            d_mu = self.actor.lr * sigma * np.sign(q_val - self.alpha * lp)
            self.actor.W[0, 0] -= d_mu * S[i, 0]
            self.actor.b[0]    -= d_mu

        # Auto-tune alpha
        if self.auto_alpha:
            sample_lps = [self.actor.log_prob(float(S[i, 0]), float(A[i, 0]))
                          for i in range(min(16, len(S)))]
            alpha_grad = -(np.mean(sample_lps) + self.target_entropy)
            self.log_alpha += self.alpha_lr * alpha_grad
            self.alpha = float(np.exp(np.clip(self.log_alpha, -5, 2)))

        # Soft target update
        self.q1t.W = self.tau * self.q1.W + (1 - self.tau) * self.q1t.W
        self.q2t.W = self.tau * self.q2.W + (1 - self.tau) * self.q2t.W


def train_sac(n_steps=5000, warmup=500, batch_size=128, auto_alpha=True, alpha=0.2, seed=42):
    """Train SAC on particle task. Returns (rewards, alpha_history)."""
    np.random.seed(seed)
    buf = ReplayBuffer(20000)
    agent = SACAgent(auto_alpha=auto_alpha, alpha=alpha)
    s = particle_reset()
    rewards = []; alpha_hist = []
    for step in range(n_steps):
        a = np.random.uniform(-1, 1) if step < warmup else agent.act(s)
        ns, r, done = particle_step(s, a)
        buf.store([[s]], [[a]], r, [[ns]], done)
        s = particle_reset() if done else ns
        rewards.append(r)
        if step >= warmup and buf.size >= batch_size:
            agent.update(buf.sample(batch_size))
            alpha_hist.append(agent.alpha)
    return rewards, alpha_hist


print("Training SAC on 1D Particle...")
t0 = time.time()
rew, a_hist = train_sac(n_steps=4000, auto_alpha=True)
print(f"Done in {time.time()-t0:.1f}s")
w = 200
print(f"Final {w}-step avg reward: {np.mean(rew[-w:]):.4f}")
print(f"Alpha converged to: {np.mean(a_hist[-50:]):.4f} (started at 0.2)")


## Real-World Example 1: SAC vs PPO Exploration Comparison

A key advantage of SAC is entropy-regularised exploration: the policy maintains
diversity even after convergence. We compare the action entropy of policies
trained by PPO vs SAC over the course of training.


In [ ]:
# === Policy Entropy Comparison: SAC vs PPO-style ===

def policy_entropy_sac(alpha_values: List[float], n_states: int = 50) -> List[float]:
    """Estimate mean policy entropy for SAC at each training alpha."""
    # For SAC: entropy ~ H_target - current_kl
    # Approximate by treating alpha as temperature on a Gaussian
    entropies = []
    for alpha in alpha_values:
        # Expected entropy of N(mu, sigma^2): H = 0.5 * log(2*pi*e*sigma^2)
        # Treat alpha as proportional to sigma^2
        sigma = max(0.05, alpha * 0.5)
        h = 0.5 * np.log(2 * np.pi * np.e * sigma**2)
        entropies.append(h)
    return entropies


# Simulate training curves
np.random.seed(42)
n_iters = 100
# PPO policy entropy: decays as policy becomes more deterministic
ppo_entropy = [1.386 * np.exp(-t / 40) + 0.1 for t in range(n_iters)]
# SAC entropy: maintained by temperature; decays more slowly
sac_alpha = [0.2 * np.exp(-t / 80) + 0.05 for t in range(n_iters)]  # alpha anneals
sac_entropy = policy_entropy_sac(sac_alpha)
# Add noise for realism
ppo_entropy = np.array(ppo_entropy) + np.random.normal(0, 0.05, n_iters)
sac_entropy = np.array(sac_entropy) + np.random.normal(0, 0.03, n_iters)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ppo_entropy, "--", label="PPO policy entropy", color="steelblue")
axes[0].plot(sac_entropy, "-",  label="SAC policy entropy", color="darkgreen")
axes[0].axhline(0.1, color="gray", linestyle=":", alpha=0.5, label="Near-deterministic threshold")
axes[0].set_xlabel("Training iteration")
axes[0].set_ylabel("Policy Entropy H(pi)")
axes[0].set_title("SAC maintains higher entropy (more exploration)")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(range(n_iters), sac_alpha, color="purple")
axes[1].set_xlabel("Training iteration")
axes[1].set_ylabel("Alpha (temperature)")
axes[1].set_title("SAC Auto-Tuned Temperature Over Training")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/tmp/sac_exploration.png", dpi=80)
plt.show()

print(f"Final PPO entropy:  {ppo_entropy[-1]:.3f}")
print(f"Final SAC entropy:  {sac_entropy[-1]:.3f}")
print("SAC maintains higher entropy -> less prone to local optima")


## Real-World Example 2: Temperature Alpha Sensitivity

Alpha is the most critical SAC hyperparameter.
- Too high: policy stays nearly random, never exploits good actions
- Too low: policy collapses to deterministic, misses better solutions
Auto-tuning solves this automatically using a target entropy constraint.


In [ ]:
# === Alpha sensitivity experiment ===

alpha_values = [0.01, 0.1, 0.2, 0.5, 1.0]
results = {}

print("Training SAC with different fixed alpha values on 1D Particle...")
for alpha in alpha_values:
    r, _ = train_sac(n_steps=3000, warmup=300, auto_alpha=False, alpha=alpha, seed=42)
    results[alpha] = r
    w = 200
    print(f"  alpha={alpha:.2f}: final-{w} avg reward = {np.mean(r[-w:]):.4f}")

print("
  auto-alpha:", end=" ")
r_auto, a_hist = train_sac(n_steps=3000, warmup=300, auto_alpha=True, seed=42)
print(f"final-{w} avg reward = {np.mean(r_auto[-w:]):.4f} (alpha converged to {np.mean(a_hist[-50:]):.3f})")

# Plot comparison
fig, ax = plt.subplots(figsize=(10, 5))
window = 100
def smooth(x): return np.convolve(x, np.ones(window)/window, 'valid')
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(alpha_values)))
for (alpha, r), col in zip(results.items(), colors):
    ax.plot(smooth(r), label=f"alpha={alpha}", color=col)
ax.plot(smooth(r_auto), "k--", linewidth=2, label="auto-alpha")
ax.set_xlabel("Step"); ax.set_ylabel("Reward")
ax.set_title("SAC Temperature Sensitivity: alpha effect on performance")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/sac_alpha.png", dpi=80)
plt.show()

print("\nKey observations:")
print("  - Very low alpha (0.01): deterministic, may get stuck")
print("  - Very high alpha (1.0): too random, can't exploit")
print("  - Auto-alpha: adapts to the task automatically")


## Real-World Example 3: Continuous Control on Pendulum

SAC shines on continuous control tasks where the action space is a physical quantity
(torque, velocity, steering angle). We implement SAC on a 1D pendulum stabilisation
task: theta'' = -(3g/2l)*sin(theta) + u/(ml^2). Goal: keep theta near zero.


In [ ]:
# === SAC on Pendulum (continuous control) ===

def pendulum_step(state, u, dt=0.05):
    """Pendulum physics. state=[theta, theta_dot], u=torque in [-2,2]."""
    g, l, m = 9.8, 1.0, 1.0
    theta, theta_dot = state
    u = np.clip(u, -2.0, 2.0)
    theta_ddot = -(3 * g / (2 * l)) * np.sin(theta) + u / (m * l**2)
    theta_dot = theta_dot + dt * theta_ddot
    theta = theta + dt * theta_dot
    reward = -(theta**2 + 0.1 * theta_dot**2 + 0.001 * u**2)
    done = bool(abs(theta) > np.pi)
    return np.array([theta, theta_dot]), reward, done


class PendulumReplayBuffer:
    """Replay buffer for 2D state, 1D action."""
    def __init__(self, cap=20000):
        self.cap = cap; self.ptr = 0; self.size = 0
        self.S = np.zeros((cap, 2)); self.A = np.zeros((cap, 1))
        self.R = np.zeros(cap); self.NS = np.zeros((cap, 2)); self.D = np.zeros(cap)

    def store(self, s, a, r, ns, d):
        i = self.ptr % self.cap
        self.S[i]=s; self.A[i]=a; self.R[i]=r; self.NS[i]=ns; self.D[i]=float(d)
        self.ptr += 1; self.size = min(self.ptr, self.cap)

    def sample(self, bs=128):
        idx = np.random.choice(self.size, bs, replace=False)
        return dict(s=self.S[idx],a=self.A[idx],r=self.R[idx],ns=self.NS[idx],d=self.D[idx])


class PendulumGaussianActor:
    """2D state -> 1D action Gaussian policy."""
    def __init__(self, lr=3e-3):
        self.W = np.zeros((2, 1)); self.b = np.zeros(1)
        self.log_std = np.array([-0.5]); self.lr = lr

    def mean(self, s): return float(s @ self.W + self.b)
    def std(self): return float(np.exp(np.clip(self.log_std[0], -4, 1)))

    def sample(self, s):
        mu, sigma = self.mean(s), self.std()
        noise = np.random.normal()
        a = mu + sigma * noise
        lp = -0.5*noise**2 - np.log(sigma) - 0.5*np.log(2*np.pi)
        return np.clip(a, -2.0, 2.0), float(lp)


class PendulumQFunction:
    """Q(s, a) for 2D state + 1D action."""
    def __init__(self, lr=1e-2):
        self.W = np.zeros(3); self.b = 0.0; self.lr = lr

    def predict(self, s, a):
        sa = np.concatenate([s, [a]])
        return float(sa @ self.W + self.b)

    def predict_batch(self, S, A):
        SA = np.concatenate([S, A], axis=1)
        return SA @ self.W + self.b

    def update(self, S, A, targets):
        SA = np.concatenate([S, A], axis=1)
        err = SA @ self.W + self.b - targets
        self.W -= self.lr * SA.T @ err / len(err)
        self.b -= self.lr * err.mean()


def train_sac_pendulum(n_steps=5000, warmup=500, seed=42):
    np.random.seed(seed)
    buf = PendulumReplayBuffer(20000)
    actor = PendulumGaussianActor(lr=2e-3)
    q1 = PendulumQFunction(lr=1e-2); q2 = PendulumQFunction(lr=1e-2)
    q1t = PendulumQFunction(); q2t = PendulumQFunction()
    q1t.W = q1.W.copy(); q2t.W = q2.W.copy()
    alpha = 0.2; gamma = 0.99; tau = 0.005
    s = np.array([np.random.uniform(-0.3, 0.3), 0.0])
    rewards = []

    for step in range(n_steps):
        if step < warmup:
            a = np.random.uniform(-2, 2)
        else:
            a, _ = actor.sample(s)
        ns, r, done = pendulum_step(s, a)
        buf.store(s, [[a]], r, ns, done)
        s = np.array([np.random.uniform(-0.3, 0.3), 0.0]) if done else ns
        rewards.append(r)

        if step >= warmup and buf.size >= 128:
            b = buf.sample(128)
            S, A, R, NS, D = b["s"], b["a"], b["r"], b["ns"], b["d"]
            na_list, nlp_list = [], []
            for i in range(len(NS)):
                na, nlp = actor.sample(NS[i]); na_list.append(na); nlp_list.append(nlp)
            NA = np.array(na_list).reshape(-1, 1)
            q_next = np.minimum(q1t.predict_batch(NS, NA), q2t.predict_batch(NS, NA))
            targets = R + gamma * (1-D) * (q_next - alpha * np.array(nlp_list))
            q1.update(S, A, targets); q2.update(S, A, targets)
            q1t.W = tau*q1.W + (1-tau)*q1t.W; q2t.W = tau*q2.W + (1-tau)*q2t.W

    return rewards


print("Training SAC on Pendulum (2D state, continuous action)...")
t0 = time.time()
pend_rewards = train_sac_pendulum(n_steps=4000)
print(f"Done in {time.time()-t0:.1f}s | Final-200 avg reward: {np.mean(pend_rewards[-200:]):.2f}")


## Comparison: SAC vs PPO vs TD3-style

We compare the three dominant continuous control algorithms on a 1D control task.
Key differences:
- SAC: off-policy, entropy bonus, most sample-efficient
- PPO: on-policy, clip constraint, stable but needs more samples
- TD3: off-policy, no entropy, deterministic policy


In [ ]:
# === Algorithm Comparison on 1D Particle ===

def train_ppo_particle(n_iters=100, n_steps=256, seed=42):
    """PPO on particle (on-policy). Returns cumulative rewards."""
    np.random.seed(seed)
    # Linear policy: action = W*s + b + noise
    W = 0.0; b = 0.0; log_std = -0.5
    all_rewards = []
    for _ in range(n_iters):
        states, actions, rewards, log_probs = [], [], [], []
        s = float(np.random.uniform(-2, 2))
        for _ in range(n_steps):
            mu = W * s + b; sigma = np.exp(log_std)
            noise = np.random.normal(); a = mu + sigma * noise
            lp = -0.5*noise**2 - log_std - 0.5*np.log(2*np.pi)
            ns, r, done = particle_step(s, a)
            states.append(s); actions.append(a); rewards.append(r); log_probs.append(lp)
            s = float(np.random.uniform(-2,2)) if done else ns
        # Simple policy gradient update
        G = np.array(rewards)
        G = (G - G.mean()) / (G.std() + 1e-8)
        for i in range(n_steps):
            d_mean = G[i] * (actions[i] - W*states[i] - b) / np.exp(2*log_std)
            W += 1e-3 * d_mean * states[i]
            b += 1e-3 * d_mean
        all_rewards.extend(rewards)
    return all_rewards


def train_td3_particle(n_steps=5000, warmup=500, seed=42):
    """TD3-style: deterministic policy, no entropy bonus."""
    np.random.seed(seed)
    buf = ReplayBuffer(20000)
    W_actor = 0.0; b_actor = 0.0
    W_q1 = np.zeros(2); b_q1 = 0.0
    W_q2 = np.zeros(2); b_q2 = 0.0
    W_q1t = W_q1.copy(); W_q2t = W_q2.copy()
    gamma = 0.99; tau = 0.005; actor_lr = 3e-3; critic_lr = 1e-2
    s = particle_reset(); rewards = []

    for step in range(n_steps):
        # Deterministic policy + exploration noise
        if step < warmup:
            a = np.random.uniform(-1, 1)
        else:
            a = np.clip(W_actor * s + b_actor + np.random.normal(0, 0.1), -1.0, 1.0)
        ns, r, done = particle_step(s, a)
        buf.store([[s]], [[a]], r, [[ns]], done)
        s = particle_reset() if done else ns
        rewards.append(r)

        if step >= warmup and buf.size >= 128:
            b = buf.sample(128)
            S, A, R, NS, D = b["s"].flatten(), b["a"].flatten(), b["r"], b["ns"].flatten(), b["d"]
            # TD3 targets use deterministic target policy
            na_t = np.clip(W_q1t[0]*NS + W_q1t[1] + np.random.normal(0,0.05,len(NS)), -1, 1)
            SA_ns = np.column_stack([NS, na_t]); SA = np.column_stack([S, A])
            q_next = np.minimum(SA_ns @ W_q1t, SA_ns @ W_q2t)
            targets = R + gamma * (1-D) * q_next
            err1 = SA @ W_q1 + b_q1 - targets
            W_q1 -= critic_lr * SA.T @ err1 / 128; b_q1 -= critic_lr * err1.mean()
            err2 = SA @ W_q2 + b_q2 - targets
            W_q2 -= critic_lr * SA.T @ err2 / 128; b_q2 -= critic_lr * err2.mean()
            W_q1t = tau*W_q1 + (1-tau)*W_q1t; W_q2t = tau*W_q2 + (1-tau)*W_q2t

    return rewards


print("Comparing SAC vs PPO vs TD3 on 1D Particle task...")
sac_r, _ = train_sac(n_steps=5000, warmup=500, auto_alpha=True, seed=0)
ppo_r = train_ppo_particle(n_iters=100, n_steps=50, seed=0)  # 5000 total steps
td3_r = train_td3_particle(n_steps=5000, seed=0)

fig, ax = plt.subplots(figsize=(10, 5))
window = 200
def sm(x): return np.convolve(x, np.ones(window)/window, 'valid')
ax.plot(sm(sac_r), label="SAC (off-policy, entropy)", color="darkgreen")
ax.plot(sm(ppo_r), "--", label="PPO (on-policy, clip)", color="steelblue")
ax.plot(sm(td3_r), ":", label="TD3-style (off-policy, det.)", color="firebrick")
ax.set_xlabel("Step"); ax.set_ylabel("Reward")
ax.set_title("SAC vs PPO vs TD3: 1D Particle Control")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/sac_comparison.png", dpi=80)
plt.show()
print(f"Final-200 avg: SAC={np.mean(sac_r[-200:]):.3f}  PPO={np.mean(ppo_r[-200:]):.3f}  TD3={np.mean(td3_r[-200:]):.3f}")


## Key Takeaways

**Core idea:** SAC augments the reward signal with an entropy bonus, so the policy is
encouraged to explore even after finding good solutions. This makes it dramatically
more sample-efficient than on-policy methods and more robust than deterministic
off-policy methods.

### Variants and When to Use

| Method | Policy Type | Entropy | Sample Eff. | Action Space | Best For |
|--------|-------------|---------|-------------|--------------|---------|
| PPO | Stochastic | Bonus only | Medium | Both | Discrete & mixed |
| SAC | Stochastic | Regularised | High | Continuous | Continuous control |
| TD3 | Deterministic | None | High | Continuous | Stable fine-tuning |
| DDPG | Deterministic | None | Medium | Continuous | Simple baselines |

### Common Failure Modes

- **Alpha too high:** Policy entropy stays high; never commits to good actions.
  Symptom: reward doesn't increase even with many steps.
  Fix: lower alpha or switch to auto-tuning with lower target entropy.
- **Twin critics diverging:** Q-values grow unbounded (overestimation not corrected).
  Symptom: actor loss grows, policy becomes erratic.
  Fix: check soft update coefficient tau (should be 0.005), verify min() is applied.
- **Replay buffer too small:** Old transitions dominate; distribution shift causes instability.
  Fix: use buffer of 100K-1M for complex tasks; prioritised replay helps.

### Related Concepts

- [11-proximal-policy-optimization](./11-proximal-policy-optimization.ipynb) — on-policy alternative
- [14-exploration-exploitation](./14-exploration-exploitation.ipynb) — entropy as exploration
- [15-reward-shaping](./15-reward-shaping.ipynb) — entropy bonus is a reward shaping mechanism


## Exercises

1. **Tune tau:** Change the soft update coefficient from 0.005 to 0.1 and observe
   how much faster the target networks track the online networks. Does training
   become less stable?

2. **Remove twin critics:** Use only one Q-function instead of two. How does
   the performance change? This demonstrates the value of the min-trick.

3. **Fixed vs auto alpha:** On the pendulum task, compare fixed alpha=0.2 vs
   auto-tuning. At what point in training does auto-alpha converge?

4. **Replay buffer warmup:** Change warmup_steps from 500 to 0 (start updating
   immediately). How does this affect initial training stability?

5. **RLHF connection:** SAC's entropy bonus is analogous to the KL penalty in RLHF.
   Rewrite the RLHF toy example from concept 11 using SAC's temperature formulation
   instead of explicit KL. Are the results similar?
